# OCR Extractor MVP

This notebook implements a lightweight OCR extraction pipeline for industrial equipment documentation. It uses a local EasyOCR engine, deterministic regex-based parsing, conservative validation, and returns a JSON dictionary without LLMs or hallucination.


In [ ]:
import json
import re
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont
import easyocr


## Initialize EasyOCR

The OCR engine runs locally in the project environment. This keeps the pipeline self-contained and avoids cloud or API-based OCR.


In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
MODEL_DIR = PROJECT_ROOT / 'model_pool' / 'easyocr'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
reader = easyocr.Reader(['en'], gpu=False, model_storage_directory=str(MODEL_DIR))
print(f'EasyOCR ready. Model dir: {MODEL_DIR}')


## OCR function

The `extract_ocr_text` helper reads the image and captures each detected text item with its confidence score. We also join the detected strings into one raw OCR text block for downstream extraction.


In [ ]:
def extract_ocr_text(file_path: str):
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f'File does not exist: {path}')

    if path.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}:
        raise ValueError(f'Unsupported file type: {path.suffix}')

    results = reader.readtext(str(path), detail=1, paragraph=False)
    detections = []
    lines = []

    for item in results:
        text = str(item[1]).strip()
        conf = float(item[2]) if len(item) > 2 else 0.0
        if text:
            detections.append({'text': text, 'confidence': conf})
            lines.append(text)

    raw_ocr_text = '\n'.join(lines)
    return {'detections': detections, 'raw_ocr_text': raw_ocr_text}


## Extraction rules

Each field uses a deterministic regex pattern and returns `None` if the relevant text is absent or invalid. This avoids hallucinating values.


In [ ]:
def normalize_whitespace(value: str) -> str:
    if value is None:
        return ''
    return re.sub(r'\s+', ' ', value).strip()


def fix_ocr_common_errors(value: str) -> str:
    if value is None:
        return ''
    value = re.sub(r'(?i)\bCorrcsion\b', 'Corrosion', value)
    value = re.sub(r'(?i)\bPate\b', 'Rate', value)
    value = re.sub(r'(?i)\bmmyr\b', 'mm/yr', value)
    value = re.sub(r'(?i)\bSeSeparator\b', 'Separator', value)
    value = re.sub(r'(?i)\bDruM\b', 'Drum', value)
    value = re.sub(r'(?i)\bI(?=\d)', '1', value)
    value = re.sub(r'(?i)(?<=\d|[A-Z])I(?=[A-Z])', '1', value)
    value = re.sub(r'(?i)\bL(?=\d)', '1', value)
    return normalize_whitespace(value).strip(' :;,.')


def extract_equipment_id(text: str):
    match = re.search(r'(?is)(?:equipment\s*id|eq\s*id)\s*[:\-]?\s*([A-Z0-9]+(?:[-/][A-Z0-9]+)+)', text)
    if not match:
        return None
    value = fix_ocr_common_errors(match.group(1))
    return value if re.search(r'\d', value) else None


def extract_equipment_name(text: str):
    match = re.search(r'(?is)(?:equipment\s*name)\s*[:\-]?\s*(.+?)(?=(?:\bmaterial\b|\bdesign\s*pressure\b|\bmeasured\s*thickness\b|\bcorrosion\s*rate\b|$))', text)
    if not match:
        return None
    value = fix_ocr_common_errors(match.group(1))
    value = value.replace('Separator', 'Separator')
    return value or None


def extract_material(text: str):
    match = re.search(r'(?is)(?:material)\s*[:\-]?\s*([A-Za-z0-9][A-Za-z0-9.\-/ ]+)', text)
    if not match:
        return None
    value = fix_ocr_common_errors(match.group(1))
    return value or None


def convert_pressure_to_mpa(value: float, unit: str) -> float | None:
    unit = (unit or '').lower()
    if unit in {'mpa', 'mpa.'}:
        return value
    if unit == 'bar':
        return value * 0.1
    if unit == 'psi':
        return value * 0.00689476
    return None


def extract_design_pressure_mpa(text: str):
    match = re.search(r'(?is)(?:design\s*pressure|pressure)\s*[:\-]?\s*([-+]?\d+(?:\.\d+)?)\s*(MPa|bar|psi)?', text)
    if not match:
        return None
    value = float(match.group(1))
    unit = (match.group(2) or 'MPa').lower()
    converted = convert_pressure_to_mpa(value, unit)
    return round(converted, 3) if converted is not None else None


def extract_measured_thickness_mm(text: str):
    match = re.search(r'(?is)(?:measured\s*thickness)\s*[:\-]?\s*([-+]?\d+(?:\.\d+)?)\s*(mm|cm|m)?', text)
    if not match:
        return None
    value = float(match.group(1))
    unit = (match.group(2) or 'mm').lower()
    if unit == 'cm':
        value *= 10
    elif unit == 'm':
        value *= 1000
    return round(value, 3)


def extract_corrosion_rate_mm_yr(text: str):
    match = re.search(r'(?is)(?:corrosion\s*rate|corrcsion\s*pate)\s*[:\-.]?\s*([-+]?\d+(?:\.\d+)?)\s*(?:mm\s*(?:/|per)?\s*yr|mmyr|mm\s*yr|mm/yr)?', text)
    if not match:
        return None
    return round(float(match.group(1)), 3)


## Validation

Basic sanity checks ensure that impossible values are set to `None` instead of being passed along.


In [ ]:
def validate_output(data):
    if not isinstance(data, dict):
        return {}

    validated = dict(data)

    for key in ['equipment_id', 'equipment_name', 'material']:
        if validated.get(key) in ('', 'N/A', 'None'):
            validated[key] = None

    pressure = validated.get('design_pressure_mpa')
    if pressure is None or not isinstance(pressure, (int, float)) or pressure <= 0:
        validated['design_pressure_mpa'] = None

    thickness = validated.get('measured_thickness_mm')
    if thickness is None or not isinstance(thickness, (int, float)) or thickness <= 0:
        validated['measured_thickness_mm'] = None

    corrosion = validated.get('corrosion_rate_mm_yr')
    if corrosion is None or not isinstance(corrosion, (int, float)) or corrosion < 0:
        validated['corrosion_rate_mm_yr'] = None

    return validated


## Confidence calculation

This uses the average EasyOCR detection confidence, not an LLM or model-based confidence estimate.


In [ ]:
def calculate_confidence(ocr_results):
    if not ocr_results:
        return 0.0
    scores = [float(item.get('confidence', 0.0)) for item in ocr_results if isinstance(item, dict)]
    if not scores:
        return 0.0
    return round(max(0.0, min(1.0, sum(scores) / len(scores))), 2)


## Final OCR tool

The final function orchestrates: file validation, OCR, field extraction, validation, confidence calculation, and output composition.


In [ ]:
def ocr_extractor_tool(file_path: str) -> dict:
    result = {
        'equipment_id': None,
        'equipment_name': None,
        'material': None,
        'design_pressure_mpa': None,
        'measured_thickness_mm': None,
        'corrosion_rate_mm_yr': None,
        'raw_ocr_text': '',
        'confidence_score': 0.0,
    }

    path = Path(file_path)
    if not path.exists():
        result['raw_ocr_text'] = f'File not found: {file_path}'
        return result

    if path.suffix.lower() not in {'.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'}:
        result['raw_ocr_text'] = f'Unsupported file type: {path.suffix}'
        return result

    try:
        ocr_output = extract_ocr_text(str(path))
        raw_text = ocr_output.get('raw_ocr_text', '')
        result['raw_ocr_text'] = raw_text
        result['equipment_id'] = extract_equipment_id(raw_text)
        result['equipment_name'] = extract_equipment_name(raw_text)
        result['material'] = extract_material(raw_text)
        result['design_pressure_mpa'] = extract_design_pressure_mpa(raw_text)
        result['measured_thickness_mm'] = extract_measured_thickness_mm(raw_text)
        result['corrosion_rate_mm_yr'] = extract_corrosion_rate_mm_yr(raw_text)

        result = validate_output(result)
        result['confidence_score'] = calculate_confidence(ocr_output.get('detections', []))
        return result
    except Exception as exc:
        result['raw_ocr_text'] = f'OCR processing failed: {exc}'
        return result


## Synthetic sample images

We generate simple images locally so the OCR pipeline can be tested without downloading external data.


In [ ]:
def create_synthetic_inspection_image(path: str, pressure_text: str = '14.5 MPa', missing_field: str | None = None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    lines = [
        'INSPECTION REPORT',
        '',
        'Equipment ID: 11-V-102',
        'Equipment Name: HP Separator Drum',
        'Material: 2.25Cr-1Mo',
        f'Design Pressure: {pressure_text}',
        'Measured Thickness: 138.2 mm',
        'Corrosion Rate: 0.75 mm/yr',
    ]

    if missing_field:
        lines = [line for line in lines if not line.lower().startswith(missing_field.lower())]

    img = Image.new('RGB', (2000, 1200), 'white')
    draw = ImageDraw.Draw(img)
    font = ImageFont.truetype(r'C:\Windows\Fonts\arial.ttf', 54)

    y = 60
    for line in lines:
        draw.text((90, y), line, fill='black', font=font, stroke_width=1, stroke_fill='black')
        y += 90

    img.save(path, format='PNG')
    return str(path)

sample_dir = PROJECT_ROOT / 'sample_data'
sample_dir.mkdir(parents=True, exist_ok=True)
normal_image = create_synthetic_inspection_image(sample_dir / 'inspection_report_test.png')
missing_field_image = create_synthetic_inspection_image(sample_dir / 'inspection_report_missing_field.png', missing_field='Equipment Name')
pressure_conversion_image = create_synthetic_inspection_image(sample_dir / 'inspection_report_pressure_bar.png', pressure_text='145 bar')
print(normal_image)
print(missing_field_image)
print(pressure_conversion_image)


## Run the tool on sample inputs

These examples validate the normal-case response and the edge cases.


In [ ]:
normal_result = ocr_extractor_tool(normal_image)
print('NORMAL RESULT')
print(json.dumps(normal_result, indent=2, ensure_ascii=False))

missing_result = ocr_extractor_tool(missing_field_image)
print('MISSING FIELD RESULT')
print(json.dumps(missing_result, indent=2, ensure_ascii=False))

bar_result = ocr_extractor_tool(pressure_conversion_image)
print('BAR PRESSURE RESULT')
print(json.dumps(bar_result, indent=2, ensure_ascii=False))


## Basic testing

The workbook checks the required scenarios: normal extraction, missing-field handling, and pressure conversion.


In [ ]:
# Test 1: normal document
normal_result = ocr_extractor_tool(normal_image)
assert normal_result['equipment_id'] == '11-V-102', normal_result
assert normal_result['equipment_name'] == 'HP Separator Drum', normal_result
assert normal_result['material'] == '2.25Cr-1Mo', normal_result
assert abs(normal_result['design_pressure_mpa'] - 14.5) < 0.5, normal_result
assert abs(normal_result['measured_thickness_mm'] - 138.2) < 0.5, normal_result
assert abs(normal_result['corrosion_rate_mm_yr'] - 0.75) < 0.2, normal_result
print('Test 1 passed: normal document extraction')

# Test 2: missing field should stay None
missing_result = ocr_extractor_tool(missing_field_image)
assert missing_result['equipment_name'] is None, missing_result
print('Test 2 passed: missing field becomes None')

# Test 3: pressure conversion from bar to MPa
bar_result = ocr_extractor_tool(pressure_conversion_image)
assert abs(bar_result['design_pressure_mpa'] - 14.5) < 0.5, bar_result
print('Test 3 passed: bar-to-MPa conversion is correct')

print('All tests passed successfully.')


## Limitations and next steps

This is intentionally a minimal MVP. It depends on local OCR quality and fixed label formats, so it may need additional rule tuning for noisy scans, rotated documents, or non-standard notation.
